# Train Models Using LeRobot on MI300x

This guide walks you through setting up environment for training imitation learning policies using LeRobot library on a DigitalOcean (DO) instance equipped with AMD MI300x GPUs and ROCm.

## ⚙️ Requirements
- A Hugging Face dataset repo ID containing your training data (`--dataset.repo_id=${HF_USER}/${DATASET_NAME}`).
  If you don’t have an access token yet, you can sign up for Hugging Face [here](https://huggingface.co/join). After signing up, create an access token by visiting [here](https://huggingface.co/settings/tokens).
- A wandb account to enable training visualization and upload your training evidence to our github.
  You can sign up for Wandb [here](https://wandb.ai/signup) and visit [here](https://wandb.ai/authorize) to create a token.
- Access to DO instance AMD Mi300x GPU


## Verify ROCm and GPU availability
This cell uses `pytorch` to check AMD GPU Info. The expected ouput is 
```
CUDA compatible device availability: True
device name [0]: AMD Instinct MI300X VF
```

In [ ]:
import torch
print(f'CUDA compatible device availability:',torch.cuda.is_available())
print(f'device name [0]:', torch.cuda.get_device_name(0))

## Install FFmpeg 7.x
This cell uses `apt` to install ffmpeg 7.x for LeRobot.

In [2]:
!add-apt-repository ppa:ubuntuhandbook1/ffmpeg7 -y # install PPA which contains ffmpeg 7.x
!apt update && apt install ffmpeg -y

Repository: 'Types: deb
URIs: https://ppa.launchpadcontent.net/ubuntuhandbook1/ffmpeg7/ubuntu/
Suites: noble
Components: main
'
Description:
unofficial build for FFmpeg 7 for Ubuntu 22.04 | 24.04, backport from Debian's deb.multimedia.org repository

If the packages here are helpful, you may buy me a coffee:

         https://ko-fi.com/ubuntuhandbook1
More info: https://launchpad.net/~ubuntuhandbook1/+archive/ubuntu/ffmpeg7
Adding repository.
Hit:1 https://repo.radeon.com/amdgpu/30.10/ubuntu jammy InRelease
Hit:2 https://repo.radeon.com/rocm/apt/7.0 jammy InRelease                     
Hit:3 https://repo.radeon.com/graphics/7.0/ubuntu jammy InRelease              
Hit:4 http://archive.ubuntu.com/ubuntu noble InRelease                         
Get:5 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]        
Get:6 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]      
Get:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease [17.8 kB]


## Install LeRobot v0.4.1
This cell clones the `lerobot` repository from Hugging Face, and installs the package in editable mode. Extra Features: To install additional dependencies for training SmolVLA or Pi models, refer to the [LeRobot offical page](https://huggingface.co/docs/lerobot/index). 


In [2]:
#!git clone https://github.com/huggingface/lerobot.git
!cd lerobot && git checkout -b v0.4.1 v0.4.1 # let’s synchronize using this version
!cd lerobot && pip install -e .
!cd lerobot && pip install -e ".[pi]"

/bin/bash: line 1: cd: lerobot: No such file or directory
/bin/bash: line 1: cd: lerobot: No such file or directory
/bin/bash: line 1: cd: lerobot: No such file or directory


## Weights & Biases login
This cell install and log into Weights & Biases (wandb) to enable experiment tracking and logging.

In [ ]:
!pip install wandb
import wandb
wandb.login(key="XXX")

## Login into Hugging Face Hub

In [ ]:
from huggingface_hub import login
login(token="XXX")

## Start Training Models with LeRobot

This cell uses the lerobot-train CLI from the lerobot library to train a robot control policy.  

Make sure to adjust the following arguments to your setup:

1. `--dataset.repo_id=YOUR_HF_USERNAME/YOUR_DATASET`:  
   Replace this with the Hugging Face Hub repo ID where your dataset is stored, e.g., `lerobot/svla_so100_pickplace`.

2. `--policy.type=act`:  
   Specifies the policy configuration to use. `act` refers to [configuration_act.py](../lerobot/common/policies/act/configuration_act.py), which will automatically adapt to your dataset’s setup (e.g., number of motors and cameras).

3. `--output_dir=outputs/train/...`:  
   Directory where training logs and model checkpoints will be saved.

4. `--job_name=...`:  
   A name for this training job, used for logging and Weights & Biases.The name typically includes the model type (e.g., act, smolvla), the dataset name, and additional descriptive tags.

5. `--policy.device=cuda`:  
   Use `cuda` if training on an AMD or NVIDIA GPU. 

6. `--wandb.enable=true`:  
   Enables Weights & Biases for visualizing training progress. You must be logged in via `wandb login` before running this.

7. `--policy.push_to_hub=`:

   Enables automatic uploading of the trained policy to the Hugging Face Hub. You must specify `--policy.repo_id` (e.g., ${HF_USER}/{REPO_NAME}) if it is True.

In [ ]:
# train act from scratch
!lerobot-train \
  --dataset.repo_id=air_hockey_test4 \
  --dataset.root=/home/ee605_supercomputer/yuting/physicalAI/air_hockey_test4 \
  --batch_size=1 \
  --steps=100000 \
  --output_dir=/home/ee605_supercomputer/yuting/physicalAI/outputs/train/air-hockey-amd-act-scratch \
  --job_name=air-hockey-amd-act-scratch \
  --policy.device=cuda \
  --policy.type=act \
  --policy.push_to_hub=false \
  --save_freq=5000

INFO 2025-12-06 14:31:43 ot_train.py:163 {'batch_size': 8,
 'checkpoint_path': None,
 'dataset': {'episodes': None,
             'image_transforms': {'enable': False,
                                  'max_num_transforms': 3,
                                  'random_order': False,
                                  'tfs': {'affine': {'kwargs': {'degrees': [-5.0,
                                                                            5.0],
                                                                'translate': [0.05,
                                                                              0.05]},
                                                     'type': 'RandomAffine',
                                                     'weight': 1.0},
                                          'brightness': {'kwargs': {'brightness': [0.8,
                                                                                   1.2]},
                                                         't

In [ ]:
# fine-tune act using npaka/air-hockey-amd-act-100000
!lerobot-train \
  --dataset.repo_id=air_hockey_test4 \
  --dataset.root=/home/ee605_supercomputer/yuting/physicalAI/air_hockey_test4 \
  --policy.path=npaka/air-hockey-amd-act-100000 \
  --policy.device=cuda \
  --batch_size=1 \
  --steps=50000 \
  --output_dir=/home/ee605_supercomputer/yuting/physicalAI/outputs/train/air-hockey-amd-act-test4 \
  --job_name=air-hockey-amd-act-test4 \
  --policy.push_to_hub=false \
  --save_freq=5000

INFO 2026-11-24 13:33:04 ot_train.py:565 {'accelerator': {'activation_checkpointing': {'mode': <ActivationCheckpointingMode.NONE: 'none'>},
                 'compile': {'backend': 'inductor',
                             'enabled': False,
                             'mode': None,
                             'regional': True},
                 'ddp': {'find_unused_parameters': True,
                         'gradient_as_bucket_view': False,
                         'static_graph': False},
                 'fsdp': {'cpu_offload': False,
                          'ignored_modules': None,
                          'min_num_params': None,
                          'mixed_precision_policy': {'cast_forward_inputs': True,
                                                     'output_dtype': None,
                                                     'param_dtype': None,
                                                     'reduce_dtype': None},
                          'reshard_after_forward'

In [ ]:
# fine-tune pi0.5 using npaka/air-hockey-amd-pi05-100000
!lerobot-train \
  --dataset.repo_id=air_hockey_test4 \
  --dataset.root=/home/ee605_supercomputer/yuting/physicalAI/air_hockey_test4 \
  --policy.path=npaka/air-hockey-amd-pi05-100000 \
  --policy.device=cuda \
  --batch_size=1 \
  --steps=50000 \
  --output_dir=/home/ee605_supercomputer/yuting/physicalAI/outputs/train/air-hockey-amd-pi05-test4 \
  --job_name=air-hockey-amd-pi05-test4 \
  --policy.push_to_hub=false \
  --save_freq=5000

INFO 2026-11-23 03:04:42 ot_train.py:565 {'accelerator': {'activation_checkpointing': {'mode': <ActivationCheckpointingMode.NONE: 'none'>},
                 'compile': {'backend': 'inductor',
                             'enabled': False,
                             'mode': None,
                             'regional': True},
                 'ddp': {'find_unused_parameters': True,
                         'gradient_as_bucket_view': False,
                         'static_graph': False},
                 'fsdp': {'cpu_offload': False,
                          'ignored_modules': None,
                          'min_num_params': None,
                          'mixed_precision_policy': {'cast_forward_inputs': True,
                                                     'output_dtype': None,
                                                     'param_dtype': None,
                                                     'reduce_dtype': None},
                          'reshard_after_forward'

**Notes**:

- If using a local dataset, add `--dataset.root=/path/to/dataset`.
- Adjust `--batch_size` and `--steps` based on your hardware and dataset.
- Model checkpoints, logs, and training plots will be saved to the specified `--output_dir`
- Training progress visualized in your wandb dashboard


## Upload Checkpoints to Hugging Face
Now after training is done, upload the last checkpoint. You may refer to [here](https://github.com/huggingface/lerobot/blob/v0.4.0/README.md#add-a-pretrained-policy) for details.

In [ ]:
!huggingface-cli upload npaka/air-hockey-amd-pi05-100000 ./outputs/train/air-hockey-amd-pi05-100000/checkpoints/last/pretrained_model
# e.g. huggingface-cli upload ${HF_USER}/act_so101_3cube_1ksteps \
#  outputs/train/act_so101_3cube_1ksteps/checkpoints/last/pretrained_model

⚠️  Warning: 'huggingface-cli upload' is deprecated. Use 'hf upload' instead.
Start hashing 7 files.
Finished hashing 7 files.
Processing Files (0 / 0)      : |                  |  0.00B /  0.00B            
New Data Upload               : |                  |  0.00B /  0.00B            

  ...d_model/model.safetensors:   0%|              |  782kB /  207MB            

Processing Files (0 / 1)      :   0%|              |  782kB /  207MB, 3.90MB/s  

Processing Files (0 / 1)      :   1%|              | 1.34MB /  207MB, 3.35MB/s  
New Data Upload               :   1%|              |  563kB / 67.1MB, 1.40MB/s  

Processing Files (0 / 1)      :  12%|█▋            | 24.4MB /  207MB, 40.7MB/s  
New Data Upload               :  18%|██▍           | 23.6MB /  134MB, 39.4MB/s  

Processing Files (0 / 1)      :  33%|████▌         | 67.7MB /  207MB, 84.5MB/s  
New Data Upload               :  33%|████▋         | 66.9MB /  201MB, 83.5MB/s  


  ...zer_processor.safetensors:  36%|█████         | 2.3

## Miscs
1. Once the environment is setup, you can open a terminal session for training by navigating to `File → New Launcher → Other → Terminal`.
2. You can also upload your datasets to the container by clicking the `Upload Files` button in the left pane.

## Q&A
1. If you encounter an error like:
   ```
   FileExistsError: Output directory outputs/train/act_so101_3cube_1ksteps already exists and resume is False. Please change your output directory so that outputs/train/act_so101_3cube_1ksteps is not overwritten. 
   ```
   Remove the existing directory before proceeding:

In [ ]:
!rm -fr outputs/train/act_so101_3cube_1ksteps

2. When running models other than ACT, ensure you install the required additional dependencies for those models.

In [7]:
!pwd

/home/ee605_supercomputer/yuting/physicalAI/AMD_Robotics_Hackathon_2025_AirHockey_pi05_ROCm-main/mission2/code


In [8]:
# For smolVLA
!cd /home/ee605_supercomputer/lerobot && pip install -e ".[smolvla]"
# For Pi
!cd /home/ee605_supercomputer/lerobot && pip install -e ".[pi]"

Obtaining file:///home/ee605_supercomputer/lerobot
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached tokenizers-0.23.3-cp310-abi3-manylinux_2_17_aarch64.manylinux2014_aarch64.whl.metadata (9.8 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 28.7 MB/s  0:00:00 eta 0:00:01
Using cached tokenizers-0.23.3-cp310-abi3-manylinux_2_17_aarch64.manylinux2014_aarch64.whl (3.5 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 806.5/806.5 kB 25.6 MB/s  0:00:00
  Building editable for lerobot (pyproject.toml) ... done
  Created wheel for lerobot: filename=lerobot-0.7.1-0.editable-py3-none-any.whl size=15395 sha256=08b15493bcf1799b2f3b9c06324d5cb49a4550192dedc77a568be76b562d2cdd
  Stored

3. If you want to resume the training from last checkpoint, run the command below:

In [ ]:
!lerobot-train \
  --resume=true \
  --config_path=outputs/train/<job name>/checkpoints/last/pretrained_model/train_config.json \
  --steps=<new total steps>